# Week 7 — Few-Shot Price Predictor

Week 7 is about fine-tuning Llama 3.2 with QLoRA to predict prices (requires GPU on Colab).

This notebook demonstrates the same idea locally using **few-shot prompting** —
giving GPT-4o-mini example price pairs before asking it to predict, which mimics
what fine-tuning achieves by exposing the model to training examples.

We compare three approaches:
- **Baseline** — always guess the average price
- **Zero-shot** — ask the LLM with no examples
- **Few-shot** — give the LLM 5 examples before asking

In [ ]:
import re
import random
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
from datasets import load_dataset
from tqdm import tqdm

load_dotenv(override=True)
client = OpenAI()
MODEL = 'gpt-4o-mini'

In [ ]:
# Load and clean Amazon data

dataset = load_dataset(
    'McAuley-Lab/Amazon-Reviews-2023',
    'raw_meta_Appliances',
    split='full',
    trust_remote_code=True
)

def clean(item):
    try:
        price = float(item['price'])
    except (ValueError, TypeError):
        return None
    if not (1 <= price <= 1000):
        return None
    title = item.get('title', '').strip()
    features = ' '.join(item.get('features', []))
    text = f"{title}. {features}".strip()
    if len(text) < 30:
        return None
    return {'text': text[:600], 'price': price}

items = [clean(d) for d in dataset]
items = [i for i in items if i]
print(f'Clean items: {len(items):,}')

In [ ]:
# Split into train (examples pool) and test set

random.seed(42)
random.shuffle(items)

train = items[:200]   # pool to draw few-shot examples from
test  = items[200:250]  # 50 items to evaluate on

avg_price = sum(i['price'] for i in train) / len(train)
print(f'Train pool : {len(train)} items')
print(f'Test set   : {len(test)} items')
print(f'Avg price  : ${avg_price:.2f}')

In [ ]:
# --- Helper: format a fine-tuning style prompt/completion pair ---
# This is exactly how the data would be formatted for QLoRA fine-tuning

def format_pair(item, include_price=True):
    prompt = f"How much does this product cost?\n\n{item['text']}\n\nPrice: $"
    completion = f"{item['price']:.2f}" if include_price else ""
    return prompt, completion

# Show an example of what fine-tuning data looks like
p, c = format_pair(train[0])
print('--- PROMPT ---')
print(p)
print('--- COMPLETION ---')
print(c)

In [ ]:
# --- Prediction functions ---

SYSTEM = "You are a pricing expert. Respond with ONLY a number representing the price in USD. No symbols, no explanation."

def extract_price(raw, fallback):
    match = re.search(r'[\d.]+', raw)
    return float(match.group()) if match else fallback

def predict_zero_shot(item):
    prompt, _ = format_pair(item, include_price=False)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": prompt},
        ],
        max_tokens=10,
    )
    return extract_price(response.choices[0].message.content.strip(), avg_price)

def predict_few_shot(item, n_examples=5):
    # Pick n random training examples to show the model
    examples = random.sample(train, n_examples)
    
    # Build a conversation with examples as prior turns
    messages = [{"role": "system", "content": SYSTEM}]
    for ex in examples:
        p, c = format_pair(ex)
        messages.append({"role": "user", "content": p})
        messages.append({"role": "assistant", "content": c})
    
    # Now ask about the real item
    prompt, _ = format_pair(item, include_price=False)
    messages.append({"role": "user", "content": prompt})
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=10,
    )
    return extract_price(response.choices[0].message.content.strip(), avg_price)

def mae(preds, actuals):
    return sum(abs(p - a) for p, a in zip(preds, actuals)) / len(actuals)

In [ ]:
# --- Run evaluation ---

actuals = [i['price'] for i in test]

print('Running zero-shot predictions...')
zero_preds = [predict_zero_shot(i) for i in tqdm(test)]

print('Running few-shot predictions...')
few_preds = [predict_few_shot(i) for i in tqdm(test)]

baseline_mae = mae([avg_price] * len(test), actuals)
zero_mae     = mae(zero_preds, actuals)
few_mae      = mae(few_preds, actuals)

print()
print('=' * 45)
print(f'  Baseline (avg guess) MAE : ${baseline_mae:.2f}')
print(f'  Zero-shot LLM MAE        : ${zero_mae:.2f}')
print(f'  Few-shot LLM MAE         : ${few_mae:.2f}')
print('=' * 45)
print(f'  Zero-shot vs baseline    : {((baseline_mae - zero_mae)/baseline_mae)*100:.1f}% better')
print(f'  Few-shot  vs zero-shot   : {((zero_mae - few_mae)/zero_mae)*100:.1f}% better')
print('=' * 45)

In [ ]:
# --- Gradio UI ---

def gradio_predict(description, mode):
    if not description.strip():
        return 'Please enter a product description.'
    item = {'text': description, 'price': 0}
    price = predict_few_shot(item) if mode == 'Few-shot (5 examples)' else predict_zero_shot(item)
    return f'${price:.2f}'

with gr.Blocks(title='Few-Shot Price Predictor') as demo:
    gr.Markdown('## The Price Is Right — Few-Shot Edition')
    gr.Markdown('Paste a product description and choose zero-shot or few-shot mode.')
    with gr.Row():
        desc = gr.Textbox(lines=6, label='Product Description', placeholder='e.g. Instant Pot Duo 7-in-1 Electric Pressure Cooker...')
        mode = gr.Radio(['Zero-shot', 'Few-shot (5 examples)'], label='Mode', value='Few-shot (5 examples)')
    btn = gr.Button('Predict Price', variant='primary')
    out = gr.Textbox(label='Predicted Price')
    btn.click(fn=gradio_predict, inputs=[desc, mode], outputs=out)

demo.launch()